# Insurance Policy RAG Comparison System

This notebook implements a RAG (Retrieval-Augmented Generation) system for comparing insurance policies based on user questions.

## Features:
1. **Hybrid Search**: Combines semantic and keyword search
2. **Reranking**: Advanced reranking for better relevance
3. **Policy Comparison**: Compare multiple policies simultaneously
4. **Structured Output**: Ready for LLM prompting and analysis

## Workflow:
1. Load policy data and generate embeddings
2. Process user questions
3. Perform RAG search across all policies
4. Generate structured results for LLM analysis


In [1]:
#!pip install rank-bm25
#!pip install faiss-cpu

## 1. Setup and Dependencies


In [1]:
# Fix OpenMP library conflicts (common on macOS)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
print("✅ Environment configured!")


✅ Environment configured!


In [2]:
# Import required libraries
import json
import os
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple, Optional
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Vector search and embeddings
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# BM25 for better keyword search
from rank_bm25 import BM25Okapi
import re

# Try to import FAISS (optional)
try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False
    print("⚠️  FAISS not available. Will use sklearn for vector search.")

# Text processing
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Download NLTK data if needed
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

print("✅ All libraries imported successfully!")


⚠️  FAISS not available. Will use sklearn for vector search.
✅ All libraries imported successfully!


## 2. Load Policy Data and Generate Embeddings


In [ ]:
class PolicyRAGSystem:
    """
    RAG system for insurance policy comparison and analysis.
    """
    
    def __init__(self, embedding_model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize the RAG system.
        
        Args:
            embedding_model_name: Name of the sentence transformer model
        """
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.policy_data = []
        self.embeddings = None
        self.tfidf_vectorizer = None
        self.tfidf_matrix = None
        self.faiss_index = None
        self.nn_index = None  # sklearn NearestNeighbors index
        self.policies = {}
        
        print(f"✅ RAG System initialized with model: {embedding_model_name}")
    
    def load_policy_data(self, data_path: str = "processed_data/embedding_texts.json"):
        """
        Load policy data from JSON file.
        
        Args:
            data_path: Path to the policy data JSON file
        """
        if not os.path.exists(data_path):
            raise FileNotFoundError(f"Policy data file not found: {data_path}")
        
        with open(data_path, 'r', encoding='utf-8') as f:
            self.policy_data = json.load(f)
        
        # Group data by policy
        for item in self.policy_data:
            policy_id = item['document_id']
            if policy_id not in self.policies:
                self.policies[policy_id] = {
                    'policy_name': item['document_name'],
                    'chunks': []
                }
            self.policies[policy_id]['chunks'].append(item)
        
        print(f"✅ Loaded {len(self.policy_data)} chunks from {len(self.policies)} policies")
        for policy_id, policy_info in self.policies.items():
            print(f"   Policy {policy_id} ({policy_info['policy_name']}): {len(policy_info['chunks'])} chunks")
    
    def generate_embeddings(self):
        """
        Generate embeddings for all policy chunks.
        """
        print("🔄 Generating embeddings...")
        
        # Extract embedding texts
        embedding_texts = [item['embedding_text'] for item in self.policy_data]
        
        # Generate embeddings
        self.embeddings = self.embedding_model.encode(embedding_texts, show_progress_bar=True)
        
        # Try to create FAISS index if available
        try:
            import faiss
            if HAS_FAISS and faiss is not None:
                dimension = self.embeddings.shape[1]
                self.faiss_index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity
                
                # Normalize embeddings for cosine similarity
                embeddings_normalized = self.embeddings.copy()
                faiss.normalize_L2(embeddings_normalized)
                self.faiss_index.add(embeddings_normalized.astype('float32'))
                
                print(f"✅ Generated embeddings: {self.embeddings.shape}")
                print(f"✅ Created FAISS index with {self.faiss_index.ntotal} vectors")
            else:
                raise ImportError("FAISS not available")
        except (ImportError, AttributeError):
            print("⚠️  FAISS not available, using sklearn NearestNeighbors")
            # Fallback to sklearn NearestNeighbors
            from sklearn.neighbors import NearestNeighbors
            from sklearn.preprocessing import normalize
            
            # Normalize embeddings for cosine similarity
            embeddings_normalized = normalize(self.embeddings, norm='l2')
            
            # Create NearestNeighbors index
            self.nn_index = NearestNeighbors(n_neighbors=len(self.embeddings), metric='cosine', algorithm='brute')
            self.nn_index.fit(embeddings_normalized)
            
            self.embeddings = embeddings_normalized
            
            print(f"✅ Generated embeddings: {self.embeddings.shape}")
            print(f"✅ Created sklearn NearestNeighbors index with {len(embeddings_normalized)} vectors")
    
    def setup_tfidf(self):
        """
        Setup TF-IDF for keyword-based search.
        """
        print("🔄 Setting up TF-IDF...")
        
        # Extract texts for TF-IDF
        texts = [item['embedding_text'] for item in self.policy_data]
        
        # Create TF-IDF vectorizer
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=10000,
            stop_words='english',
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.8
        )
        
        # Fit and transform
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(texts)
        
        print(f"✅ TF-IDF setup complete: {self.tfidf_matrix.shape}")



# Initialize RAG system
rag_system = PolicyRAGSystem()


✅ RAG System initialized with model: all-MiniLM-L6-v2


In [5]:
# Add BM25 support to PolicyRAGSystem
# Add these attributes to __init__
PolicyRAGSystem._init_original = PolicyRAGSystem.__init__

def __init_bm25(self, embedding_model_name: str = 'all-MiniLM-L6-v2'):
    """Modified __init__ with BM25 support."""
    self._init_original(embedding_model_name)
    self.bm25 = None
    self.bm25_tokenized_corpus = None

PolicyRAGSystem.__init__ = __init_bm25

# Helper method for tokenization
def _tokenize_bm25(self, text: str):
    """Tokenize text for BM25."""
    import re
    tokens = re.findall(r'\b\w+\b', text.lower())
    return tokens
PolicyRAGSystem._tokenize = _tokenize_bm25

# Add setup_bm25 method
def setup_bm25(self):
    """Setup BM25 for better keyword-based search."""
    print("🔄 Setting up BM25...")
    
    # Extract texts for BM25
    texts = [item['embedding_text'] for item in self.policy_data]
    
    # Tokenize corpus for BM25
    self.bm25_tokenized_corpus = [self._tokenize(text) for text in texts]
    
    # Create and fit BM25
    self.bm25 = BM25Okapi(self.bm25_tokenized_corpus)
    
    print(f"✅ BM25 setup complete with {len(self.bm25_tokenized_corpus)} documents")

PolicyRAGSystem.setup_bm25 = setup_bm25

print("✅ BM25 support added to PolicyRAGSystem!")

✅ BM25 support added to PolicyRAGSystem!


In [6]:
# Load policy data and generate embeddings
rag_system.load_policy_data()
rag_system.generate_embeddings()
rag_system.setup_tfidf()
rag_system.setup_bm25()


✅ Loaded 90 chunks from 2 policies
   Policy 0 (policy1.pdf): 52 chunks
   Policy 1 (policy2.pdf): 38 chunks
🔄 Generating embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

⚠️  FAISS not available, using sklearn NearestNeighbors
✅ Generated embeddings: (90, 384)
✅ Created sklearn NearestNeighbors index with 90 vectors
🔄 Setting up TF-IDF...
✅ TF-IDF setup complete: (90, 3594)
🔄 Setting up BM25...
✅ BM25 setup complete with 90 documents


## 3. RAG Search Methods


In [7]:
class RAGSearchMethods:
    """
    Different RAG search methods for policy comparison.
    """
    
    def __init__(self, rag_system: PolicyRAGSystem):
        self.rag_system = rag_system
    
    def semantic_search(self, query: str, top_k: int = 10) -> List[Dict[str, Any]]:
        """
        Perform semantic search using embeddings.
        
        Args:
            query: Search query
            top_k: Number of top results to return
            
        Returns:
            List of search results with metadata
        """
        # Generate query embedding
        query_embedding = self.rag_system.embedding_model.encode([query])
        
        # Try to use FAISS if available
        if self.rag_system.faiss_index is not None:
            try:
                import faiss
                faiss.normalize_L2(query_embedding)
                
                # Search using FAISS
                actual_top_k = min(top_k, len(self.rag_system.policy_data))
                scores, indices = self.rag_system.faiss_index.search(
                    query_embedding.astype('float32'), actual_top_k
                )
                
                results = []
                for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
                    if 0 <= idx < len(self.rag_system.policy_data):
                        item = self.rag_system.policy_data[idx]
                        results.append({
                            'rank': i + 1,
                            'chunk_id': item['chunk_id'],
                            'policy_id': item['document_id'],
                            'policy_name': item['document_name'],
                            'text': item['original_text'],
                            'embedding_text': item['embedding_text'],
                            'semantic_score': float(score),
                            'metadata': item['metadata']
                        })
                return results
            except ImportError:
                pass
        
        # Fallback to sklearn or direct cosine similarity
        if self.rag_system.nn_index is not None:
            # Use sklearn NearestNeighbors
            from sklearn.preprocessing import normalize
            query_normalized = normalize(query_embedding, norm='l2')
            
            actual_top_k = min(top_k, len(self.rag_system.policy_data))
            distances, indices = self.rag_system.nn_index.kneighbors(
                query_normalized, n_neighbors=actual_top_k
            )
            
            results = []
            for i, (distance, idx) in enumerate(zip(distances[0], indices[0])):
                if 0 <= idx < len(self.rag_system.policy_data):
                    item = self.rag_system.policy_data[idx]
                    similarity_score = 1 - float(distance)  # Convert distance to similarity
                    results.append({
                        'rank': i + 1,
                        'chunk_id': item['chunk_id'],
                        'policy_id': item['document_id'],
                        'policy_name': item['document_name'],
                        'text': item['original_text'],
                        'embedding_text': item['embedding_text'],
                        'semantic_score': similarity_score,
                        'metadata': item['metadata']
                    })
            return results
        else:
            # Direct cosine similarity (no index)
            from sklearn.metrics.pairwise import cosine_similarity
            similarities = cosine_similarity(query_embedding, self.rag_system.embeddings)[0]
            top_indices = similarities.argsort()[-min(top_k, len(similarities)):][::-1]
            
            results = []
            for i, idx in enumerate(top_indices):
                if similarities[idx] > 0:
                    item = self.rag_system.policy_data[idx]
                    results.append({
                        'rank': i + 1,
                        'chunk_id': item['chunk_id'],
                        'policy_id': item['document_id'],
                        'policy_name': item['document_name'],
                        'text': item['original_text'],
                        'embedding_text': item['embedding_text'],
                        'semantic_score': float(similarities[idx]),
                        'metadata': item['metadata']
                    })
            return results
    
    def keyword_search_tfidf(self, query: str, top_k: int = 10) -> List[Dict[str, Any]]:
        """
        Perform keyword-based search using TF-IDF.
        
        Args:
            query: Search query
            top_k: Number of top results to return
            
        Returns:
            List of search results with metadata
        """
        # Transform query
        query_vector = self.rag_system.tfidf_vectorizer.transform([query])
        
        # Calculate similarities
        similarities = cosine_similarity(query_vector, self.rag_system.tfidf_matrix)[0]
        
        # Get top results
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        results = []
        for i, idx in enumerate(top_indices):
            if similarities[idx] > 0:  # Only include results with positive similarity
                item = self.rag_system.policy_data[idx]
                results.append({
                    'rank': i + 1,
                    'chunk_id': item['chunk_id'],
                    'policy_id': item['document_id'],
                    'policy_name': item['document_name'],
                    'text': item['original_text'],
                    'embedding_text': item['embedding_text'],
                    'keyword_score': float(similarities[idx]),
                    'metadata': item['metadata']
                })
        
        return results

    def keyword_search(self, query: str, top_k: int = 10):
        """
        使用 BM25 进行关键词搜索（替代 TF-IDF）
        """
        if not query or not query.strip() or top_k <= 0:
            return []
        
        # 使用 BM25（推荐）
        if self.rag_system.bm25 is not None:
            query_tokens = re.findall(r'\b\w+\b', query.lower())
            scores = self.rag_system.bm25.get_scores(query_tokens)
            actual_top_k = min(top_k, len(scores))
            top_indices = scores.argsort()[-actual_top_k:][::-1]
            
            results = []
            for i, idx in enumerate(top_indices):
                if 0 <= idx < len(self.rag_system.policy_data) and scores[idx] > 0:
                    item = self.rag_system.policy_data[idx]
                    results.append({
                        'rank': i + 1,
                        'chunk_id': item['chunk_id'],
                        'policy_id': item['document_id'],
                        'policy_name': item['document_name'],
                        'text': item['original_text'],
                        'embedding_text': item['embedding_text'],
                        'keyword_score': float(scores[idx]),
                        'metadata': item['metadata']
                    })
            return results
        else:
            # TF-IDF 后备方案
            from sklearn.metrics.pairwise import cosine_similarity
            query_vector = self.rag_system.tfidf_vectorizer.transform([query])
            similarities = cosine_similarity(query_vector, self.rag_system.tfidf_matrix)[0]
            top_indices = similarities.argsort()[-top_k:][::-1]
            
            results = []
            for i, idx in enumerate(top_indices):
                if similarities[idx] > 0:
                    item = self.rag_system.policy_data[idx]
                    results.append({
                        'rank': i + 1,
                        'chunk_id': item['chunk_id'],
                        'policy_id': item['document_id'],
                        'policy_name': item['document_name'],
                        'text': item['original_text'],
                        'embedding_text': item['embedding_text'],
                        'keyword_score': float(similarities[idx]),
                        'metadata': item['metadata']
                    })
            return results
            
    
    def hybrid_search(self, query: str, top_k: int = 10, alpha: float = 0.7) -> List[Dict[str, Any]]:
        """
        Perform hybrid search combining semantic and keyword search.
        
        Args:
            query: Search query
            top_k: Number of top results to return
            alpha: Weight for semantic search (1-alpha for keyword search)
            
        Returns:
            List of search results with combined scores
        """
        # Get results from both methods
        semantic_results = self.semantic_search(query, top_k * 2)
        keyword_results = self.keyword_search(query, top_k * 2)
        
        # Create score dictionaries for easy lookup
        semantic_scores = {r['chunk_id']: r['semantic_score'] for r in semantic_results}
        keyword_scores = {r['chunk_id']: r['keyword_score'] for r in keyword_results}
        
        # Combine results
        combined_results = {}
        
        # Add semantic results
        for result in semantic_results:
            chunk_id = result['chunk_id']
            combined_results[chunk_id] = result.copy()
            combined_results[chunk_id]['hybrid_score'] = alpha * result['semantic_score']
            combined_results[chunk_id]['keyword_score'] = keyword_scores.get(chunk_id, 0)
        
        # Add keyword results
        for result in keyword_results:
            chunk_id = result['chunk_id']
            if chunk_id not in combined_results:
                combined_results[chunk_id] = result.copy()
                combined_results[chunk_id]['semantic_score'] = semantic_scores.get(chunk_id, 0)
                combined_results[chunk_id]['hybrid_score'] = (1 - alpha) * result['keyword_score']
            else:
                combined_results[chunk_id]['hybrid_score'] += (1 - alpha) * result['keyword_score']
        
        # Sort by hybrid score and return top_k
        sorted_results = sorted(
            combined_results.values(),
            key=lambda x: x['hybrid_score'],
            reverse=True
        )
        
        return sorted_results[:top_k]

# Initialize search methods
search_methods = RAGSearchMethods(rag_system)


In [8]:
import re

def keyword_search_bm25_patched(self, query: str, top_k: int = 10):
    """
    使用 BM25 进行关键词搜索（替代 TF-IDF）
    """
    if not query or not query.strip() or top_k <= 0:
        return []
    
    # 使用 BM25（推荐）
    if self.rag_system.bm25 is not None:
        query_tokens = re.findall(r'\b\w+\b', query.lower())
        scores = self.rag_system.bm25.get_scores(query_tokens)
        actual_top_k = min(top_k, len(scores))
        top_indices = scores.argsort()[-actual_top_k:][::-1]
        
        results = []
        for i, idx in enumerate(top_indices):
            if 0 <= idx < len(self.rag_system.policy_data) and scores[idx] > 0:
                item = self.rag_system.policy_data[idx]
                results.append({
                    'rank': i + 1,
                    'chunk_id': item['chunk_id'],
                    'policy_id': item['document_id'],
                    'policy_name': item['document_name'],
                    'text': item['original_text'],
                    'embedding_text': item['embedding_text'],
                    'keyword_score': float(scores[idx]),
                    'metadata': item['metadata']
                })
        return results
    else:
        # TF-IDF 后备方案
        from sklearn.metrics.pairwise import cosine_similarity
        query_vector = self.rag_system.tfidf_vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.rag_system.tfidf_matrix)[0]
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        results = []
        for i, idx in enumerate(top_indices):
            if similarities[idx] > 0:
                item = self.rag_system.policy_data[idx]
                results.append({
                    'rank': i + 1,
                    'chunk_id': item['chunk_id'],
                    'policy_id': item['document_id'],
                    'policy_name': item['document_name'],
                    'text': item['original_text'],
                    'embedding_text': item['embedding_text'],
                    'keyword_score': float(similarities[idx]),
                    'metadata': item['metadata']
                })
        return results

# 替换原始方法
RAGSearchMethods.keyword_search = keyword_search_bm25_patched

print("✅ keyword_search 已更新为使用 BM25！")
print("📊 BM25 提供了更好的搜索质量，特别是对罕见词的处理")

✅ keyword_search 已更新为使用 BM25！
📊 BM25 提供了更好的搜索质量，特别是对罕见词的处理


## 4. Reranking Methods


In [9]:
class RerankingMethods:
    """
    Advanced reranking methods for improving search relevance.
    """
    
    def __init__(self, rag_system: PolicyRAGSystem):
        self.rag_system = rag_system
        self.stop_words = set(stopwords.words('english'))
    
    def extract_keywords(self, text: str) -> List[str]:
        """
        Extract keywords from text.
        
        Args:
            text: Input text
            
        Returns:
            List of keywords
        """
        # Tokenize and clean
        tokens = word_tokenize(text.lower())
        keywords = [token for token in tokens 
                   if token.isalpha() and token not in self.stop_words and len(token) > 2]
        return keywords
    
    def calculate_query_coverage(self, query: str, text: str) -> float:
        """
        Calculate how well the text covers the query terms.
        
        Args:
            query: Search query
            text: Document text
            
        Returns:
            Coverage score (0-1)
        """
        query_keywords = self.extract_keywords(query)
        text_keywords = self.extract_keywords(text)
        
        if not query_keywords:
            return 0.0
        
        # Calculate coverage
        covered_keywords = set(query_keywords) & set(text_keywords)
        coverage = len(covered_keywords) / len(query_keywords)
        
        return coverage
    
    def calculate_financial_relevance(self, query: str, metadata: Dict[str, Any]) -> float:
        """
        Calculate financial relevance score based on query and metadata.
        
        Args:
            query: Search query
            metadata: Document metadata containing financial info
            
        Returns:
            Financial relevance score (0-1)
        """
        financial_terms = ['cost', 'price', 'copay', 'deductible', 'premium', 'out-of-pocket', 
                          'coinsurance', 'dollar', '$', 'cheap', 'expensive', 'affordable']
        
        query_lower = query.lower()
        has_financial_query = any(term in query_lower for term in financial_terms)
        
        if not has_financial_query:
            return 0.5  # Neutral score for non-financial queries
        
        # Check if document has financial information
        financial_info = metadata.get('financial_info', {})
        has_financial_data = (
            financial_info.get('dollar_amounts') or 
            financial_info.get('deductibles') or 
            financial_info.get('copays')
        )
        
        if has_financial_data:
            return 1.0  # High relevance for financial queries with financial data
        else:
            return 0.3  # Lower relevance for financial queries without financial data
    
    def calculate_policy_specificity(self, text: str, key_terms: List[str]) -> float:
        """
        Calculate how policy-specific the text is.
        
        Args:
            text: Document text
            key_terms: List of key terms from metadata
            
        Returns:
            Policy specificity score (0-1)
        """
        if not key_terms:
            return 0.5
        
        text_lower = text.lower()
        matched_terms = sum(1 for term in key_terms if term.lower() in text_lower)
        
        return min(matched_terms / len(key_terms), 1.0)
    
    def rerank_results(self, results: List[Dict[str, Any]], query: str, 
                      weights: Dict[str, float] = None) -> List[Dict[str, Any]]:
        """
        Rerank search results using multiple factors.
        
        Args:
            results: List of search results
            query: Original search query
            weights: Weights for different reranking factors
            
        Returns:
            Reranked results
        """
        if weights is None:
            weights = {
                'original_score': 0.4,
                'query_coverage': 0.3,
                'financial_relevance': 0.2,
                'policy_specificity': 0.1
            }
        
        reranked_results = []
        
        for result in results:
            # Calculate additional scores
            query_coverage = self.calculate_query_coverage(query, result['text'])
            financial_relevance = self.calculate_financial_relevance(
                query, result['metadata']
            )
            policy_specificity = self.calculate_policy_specificity(
                result['text'], result['metadata'].get('key_terms', [])
            )
            
            # Get original score (hybrid_score if available, otherwise semantic_score)
            original_score = result.get('hybrid_score', result.get('semantic_score', 0))
            
            # Calculate final reranked score
            reranked_score = (
                weights['original_score'] * original_score +
                weights['query_coverage'] * query_coverage +
                weights['financial_relevance'] * financial_relevance +
                weights['policy_specificity'] * policy_specificity
            )
            
            # Add reranking information
            result_copy = result.copy()
            result_copy['reranked_score'] = reranked_score
            result_copy['query_coverage'] = query_coverage
            result_copy['financial_relevance'] = financial_relevance
            result_copy['policy_specificity'] = policy_specificity
            
            reranked_results.append(result_copy)
        
        # Sort by reranked score
        reranked_results.sort(key=lambda x: x['reranked_score'], reverse=True)
        
        # Update ranks
        for i, result in enumerate(reranked_results):
            result['final_rank'] = i + 1
        
        return reranked_results

# Initialize reranking methods
reranking_methods = RerankingMethods(rag_system)


## 5. Policy Comparison Engine


In [10]:
class PolicyComparisonEngine:
    """
    Main engine for policy comparison using RAG search.
    """
    
    def __init__(self, rag_system: PolicyRAGSystem, search_methods: RAGSearchMethods, 
                 reranking_methods: RerankingMethods):
        self.rag_system = rag_system
        self.search_methods = search_methods
        self.reranking_methods = reranking_methods
    
    def search_with_hybrid(self, query: str, top_k: int = 10) -> Dict[str, Any]:
        """
        Perform hybrid search without reranking.
        
        Args:
            query: Search query
            top_k: Number of results per policy
            
        Returns:
            Structured search results
        """
        # Perform hybrid search
        results = self.search_methods.hybrid_search(query, top_k * 2)
        
        # Group results by policy
        policy_results = {}
        for result in results:
            policy_id = result['policy_id']
            if policy_id not in policy_results:
                policy_results[policy_id] = {
                    'policy_name': result['policy_name'],
                    'results': []
                }
            policy_results[policy_id]['results'].append(result)
        
        # Limit results per policy
        for policy_id in policy_results:
            policy_results[policy_id]['results'] = policy_results[policy_id]['results'][:top_k]
        
        return {
            'query': query,
            'method': 'hybrid_search',
            'total_results': len(results),
            'policies': policy_results,
            'search_metadata': {
                'top_k': top_k,
                'alpha': 0.7  # Default hybrid weight
            }
        }
    
    def search_with_rerank(self, query: str, top_k: int = 10, 
                          rerank_weights: Dict[str, float] = None) -> Dict[str, Any]:
        """
        Perform hybrid search with reranking.
        
        Args:
            query: Search query
            top_k: Number of results per policy
            rerank_weights: Weights for reranking factors
            
        Returns:
            Structured search results with reranking
        """
        # Perform hybrid search
        results = self.search_methods.hybrid_search(query, top_k * 3)
        
        # Apply reranking
        reranked_results = self.reranking_methods.rerank_results(
            results, query, rerank_weights
        )
        
        # Group results by policy
        policy_results = {}
        for result in reranked_results:
            policy_id = result['policy_id']
            if policy_id not in policy_results:
                policy_results[policy_id] = {
                    'policy_name': result['policy_name'],
                    'results': []
                }
            policy_results[policy_id]['results'].append(result)
        
        # Limit results per policy
        for policy_id in policy_results:
            policy_results[policy_id]['results'] = policy_results[policy_id]['results'][:top_k]
        
        return {
            'query': query,
            'method': 'hybrid_search_with_rerank',
            'total_results': len(reranked_results),
            'policies': policy_results,
            'search_metadata': {
                'top_k': top_k,
                'alpha': 0.7,
                'rerank_weights': rerank_weights or {
                    'original_score': 0.4,
                    'query_coverage': 0.3,
                    'financial_relevance': 0.2,
                    'policy_specificity': 0.1
                }
            }
        }
    
    def format_for_llm(self, search_results: Dict[str, Any]) -> str:
        """
        Format search results for LLM prompting.
        
        Args:
            search_results: Results from search methods
            
        Returns:
            Formatted string for LLM input
        """
        formatted_output = f"""
# Policy Comparison Results

## Query: {search_results['query']}
## Search Method: {search_results['method']}
## Total Results: {search_results['total_results']}

"""
        
        for policy_id, policy_data in search_results['policies'].items():
            policy_name = policy_data['policy_name']
            results = policy_data['results']
            
            formatted_output += f"""
## Policy: {policy_name} (ID: {policy_id})

### Top Results:
"""
            
            for i, result in enumerate(results, 1):
                score_key = 'reranked_score' if 'reranked_score' in result else 'hybrid_score'
                score = result.get(score_key, 0)
                
                formatted_output += f"""
**Result {i}** (Score: {score:.3f})
- Text: {result['text'][:200]}...
- Key Terms: {', '.join(result['metadata'].get('key_terms', [])[:5])}
- Financial Info: {result['metadata'].get('financial_info', {}).get('dollar_amounts', [])[:3]}
"""
                
                if 'reranked_score' in result:
                    formatted_output += f"""
- Query Coverage: {result['query_coverage']:.3f}
- Financial Relevance: {result['financial_relevance']:.3f}
- Policy Specificity: {result['policy_specificity']:.3f}
"""
        
        return formatted_output
    
    def extract_financial_data(self, search_results: Dict[str, Any]) -> Dict[str, Any]:
        """
        Extract financial data from search results for comparison.
        
        Args:
            search_results: Results from search methods
            
        Returns:
            Extracted financial data by policy
        """
        financial_data = {}
        
        for policy_id, policy_data in search_results['policies'].items():
            policy_name = policy_data['policy_name']
            financial_data[policy_id] = {
                'policy_name': policy_name,
                'dollar_amounts': [],
                'deductibles': [],
                'copays': [],
                'percentages': []
            }
            
            for result in policy_data['results']:
                financial_info = result['metadata'].get('financial_info', {})
                
                financial_data[policy_id]['dollar_amounts'].extend(
                    financial_info.get('dollar_amounts', [])
                )
                financial_data[policy_id]['deductibles'].extend(
                    financial_info.get('deductibles', [])
                )
                financial_data[policy_id]['copays'].extend(
                    financial_info.get('copays', [])
                )
                financial_data[policy_id]['percentages'].extend(
                    financial_info.get('percentages', [])
                )
            
            # Remove duplicates and sort
            for key in ['dollar_amounts', 'deductibles', 'copays', 'percentages']:
                financial_data[policy_id][key] = sorted(
                    list(set(financial_data[policy_id][key]))
                )
        
        return financial_data

# Initialize comparison engine
comparison_engine = PolicyComparisonEngine(rag_system, search_methods, reranking_methods)


In [11]:
# 🎯 Modified Demo with Full Text Display
# This cell replaces Cell 15 to show complete text instead of truncated

test_query = "which policy has cheaper copay"

print("="*80)
print(f"🔍 Testing Query: '{test_query}'")
print("="*80)

# Option 1: Hybrid Search (without reranking)
print("\n📊 Option 1: Hybrid Search")
print("-"*80)
hybrid_results = comparison_engine.search_with_hybrid(test_query, top_k=5)

print(f"\nFound results in {len(hybrid_results['policies'])} policies:")
for policy_id, policy_data in hybrid_results['policies'].items():
    print(f"\n  {policy_data['policy_name']}: {len(policy_data['results'])} results")
    for i, result in enumerate(policy_data['results'][:5], 1):
        print(f"    {i}. Score: {result['hybrid_score']:.3f}")
        print(f"       Text (Full):\n{result['text']}")
        print(f"       {'-'*80}")

# Option 2: Hybrid Search with Reranking
print("\n\n📊 Option 2: Hybrid Search + Reranking")
print("-"*80)
reranked_results = comparison_engine.search_with_rerank(test_query, top_k=5)

print(f"\nFound results in {len(reranked_results['policies'])} policies:")
for policy_id, policy_data in reranked_results['policies'].items():
    print(f"\n  {policy_data['policy_name']}: {len(policy_data['results'])} results")
    for i, result in enumerate(policy_data['results'][:5], 1):
        print(f"    {i}. Reranked Score: {result['reranked_score']:.3f}")
        print(f"       Original Score: {result['hybrid_score']:.3f}")
        print(f"       Query Coverage: {result['query_coverage']:.3f}")
        print(f"       Financial Relevance: {result['financial_relevance']:.3f}")
        print(f"       Policy Specificity: {result['policy_specificity']:.3f}")
        print(f"       Text (Full):\n{result['text']}")
        print(f"       {'-'*80}")

# Extract financial data for comparison
print("\n\n💰 Extracted Financial Data:")
print("-"*80)
financial_data = comparison_engine.extract_financial_data(reranked_results)
for policy_id, data in financial_data.items():
    print(f"\n{data['policy_name']}:")
    print(f"  Copays: {data['copays'][:5]}")
    print(f"  Deductibles: {data['deductibles'][:3]}")
    print(f"  Dollar amounts: {data['dollar_amounts'][:5]}")


🔍 Testing Query: 'which policy has cheaper copay'

📊 Option 1: Hybrid Search
--------------------------------------------------------------------------------

Found results in 2 policies:

  policy2.pdf: 4 results
    1. Score: 2.282
       Text (Full):
Do you need a referral to see a specialist? Yes. This plan will pay some or all of the costs to see a specialist for covered services but only if you have a referral before you see the specialist. For more information about limitations and exceptions, see the plan or policy document at www.bcbstx.combbindbb_bhsd42bftitxp_tx_2025.pdf of 8 All copayment and coinsurance costs shown in this chart are after your deductible has been met, if a deductible applies.
       --------------------------------------------------------------------------------
    2. Score: 1.074
       Text (Full):
For more information about limitations and exceptions, see the plan or policy document at www.bcbstx.combbindbb_bhsd42bftitxp_tx_2025.pdf of 8 Common Medical

## 6. Demo and Testing


In [12]:
# 🎯 Modified Additional Test Queries with Full Text Display
# This replaces the original Additional Test Queries cell to show complete text

# Test with additional queries
additional_queries = [
    "what is the deductible for each policy",
    "emergency room coverage comparison",
    "prescription drug costs",
    "out of pocket maximum"
]

all_results = {}

for query in additional_queries:
    print(f"\n{'='*80}")
    print(f"🔍 Query: '{query}'")
    print('='*80)
    
    # Get reranked results
    results = comparison_engine.search_with_rerank(query, top_k=3)
    all_results[query] = results
    
    # Show summary with FULL text
    print(f"\nResults by policy:")
    for policy_id, policy_data in results['policies'].items():
        print(f"\n  {policy_data['policy_name']}: {len(policy_data['results'])} results")
        
        # Show top result with FULL text
        if policy_data['results']:
            top = policy_data['results'][0]
            print(f"    Top score: {top['reranked_score']:.3f}")
            print(f"    Text (Full):\n{top['text']}")
            print(f"    {'-'*80}")
    
    # Extract and show financial data
    financial = comparison_engine.extract_financial_data(results)
    print(f"\n  Financial highlights:")
    for policy_id, data in financial.items():
        if data['copays'] or data['deductibles']:
            print(f"    {data['policy_name']}:")
            if data['copays']:
                print(f"      Copays: {', '.join(data['copays'][:3])}")
            if data['deductibles']:
                print(f"      Deductibles: {', '.join(data['deductibles'][:3])}")

# Save all results
with open("rag_outputs/all_queries_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\n\n✅ All query results saved to 'rag_outputs/all_queries_results.json'")



🔍 Query: 'what is the deductible for each policy'

Results by policy:

  policy2.pdf: 3 results
    Top score: 1.566
    Text (Full):
For more information about limitations and exceptions, see the plan or policy document at www.bcbstx.combbindbb_bhsd42bftitxp_tx_2025.pdf of 8 Common Medical Event Services You May Need What You Will Pay Limitations, Exceptions, Other Important Information Participating Providers (You will pay the least) Non-Participating Providers (You will pay the most) If you need drugs to treat your illness or condition More information about prescription drug coverage is available at www.bcbstx.comrx25 4T Generic drugs Retail: Preferred Participating - $25prescription Participating - $25prescription Mail - $75prescription; deductible does not apply Not Covered Limited to a 30-day supply at retail (or a 90-day supply at a network of select retail pharmacies).
    --------------------------------------------------------------------------------

  policy1.pdf: 3 resul

## 7. Generate LLM-Ready Output


In [13]:
# Generate formatted output for LLM prompting

print("="*80)
print("📝 Generating LLM-Ready Output")
print("="*80)

# Format both results for LLM
llm_output_hybrid = comparison_engine.format_for_llm(hybrid_results)
llm_output_reranked = comparison_engine.format_for_llm(reranked_results)

# Save outputs
os.makedirs("rag_outputs", exist_ok=True)

# Save hybrid search results
with open("rag_outputs/hybrid_search_results.txt", "w", encoding="utf-8") as f:
    f.write(llm_output_hybrid)

# Save reranked results
with open("rag_outputs/reranked_search_results.txt", "w", encoding="utf-8") as f:
    f.write(llm_output_reranked)

# Save JSON results for programmatic access
with open("rag_outputs/hybrid_search_results.json", "w", encoding="utf-8") as f:
    json.dump(hybrid_results, f, indent=2, ensure_ascii=False)

with open("rag_outputs/reranked_search_results.json", "w", encoding="utf-8") as f:
    json.dump(reranked_results, f, indent=2, ensure_ascii=False)

print("\n✅ Outputs saved to 'rag_outputs/' directory:")
print("   - hybrid_search_results.txt (formatted for LLM)")
print("   - reranked_search_results.txt (formatted for LLM)")
print("   - hybrid_search_results.json (structured data)")
print("   - reranked_search_results.json (structured data)")

# Display preview of reranked output
print("\n" + "="*80)
print("📄 Preview of Reranked Output for LLM:")
print("="*80)
print(llm_output_reranked[:1000] + "\n...(truncated)")


📝 Generating LLM-Ready Output

✅ Outputs saved to 'rag_outputs/' directory:
   - hybrid_search_results.txt (formatted for LLM)
   - reranked_search_results.txt (formatted for LLM)
   - hybrid_search_results.json (structured data)
   - reranked_search_results.json (structured data)

📄 Preview of Reranked Output for LLM:

# Policy Comparison Results

## Query: which policy has cheaper copay
## Search Method: hybrid_search_with_rerank
## Total Results: 15


## Policy: policy2.pdf (ID: 1)

### Top Results:

**Result 1** (Score: 1.173)
- Text: Do you need a referral to see a specialist? Yes. This plan will pay some or all of the costs to see a specialist for covered services but only if you have a referral before you see the specialist. For...
- Key Terms: specialist, coinsurance, limitation, deductible, copay
- Financial Info: []

- Query Coverage: 0.333
- Financial Relevance: 0.300
- Policy Specificity: 1.000

**Result 2** (Score: 0.702)
- Text: What is the out-of-pocket limit for this pl

## 8. Additional Test Queries


In [ ]:
# Test with additional queries
additional_queries = [
    "deductible comparison",
    "emergency room coverage comparison",
    "coverage for pregnancy and diabetes",#"compare coverage for pregnancy and diabetes between policies",
    "copmare copay for primary doctor visit"
]

all_results = {}

for query in additional_queries:
    print(f"\n{'='*80}")
    print(f"🔍 Query: '{query}'")
    print('='*80)
    
    # Get reranked results
    results = comparison_engine.search_with_rerank(query, top_k=3)
    all_results[query] = results
    
    # Show summary
    print(f"\nResults by policy:")
    for policy_id, policy_data in results['policies'].items():
        print(f"  {policy_data['policy_name']}: {len(policy_data['results'])} results")
        
        # Show top result
        if policy_data['results']:
            top = policy_data['results'][0]
            print(f"    Top score: {top['reranked_score']:.3f}")
            print(f"    Text preview: {top['text'][:120]}...")
    
    # Extract and show financial data
    financial = comparison_engine.extract_financial_data(results)
    print(f"\n  Financial highlights:")
    for policy_id, data in financial.items():
        if data['copays'] or data['deductibles']:
            print(f"    {data['policy_name']}:")
            if data['copays']:
                print(f"      Copays: {', '.join(data['copays'][:3])}")
            if data['deductibles']:
                print(f"      Deductibles: {', '.join(data['deductibles'][:3])}")

# Save all results
with open("rag_outputs/all_queries_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\n\n✅ All query results saved to 'rag_outputs/all_queries_results.json'")



🔍 Query: 'coverage for pregnancy and diabetes'

Results by policy:
  policy1.pdf: 3 results
    Top score: 1.579
    Text preview: $0 Tubal ligation $0 Vasectomy $0 Medical nutrition therapy, not related to diabetes $0 Pregnancy and maternity care Phy...
  policy2.pdf: 3 results
    Top score: 1.019
    Text preview: Abortion (Except for a pregnancy that, as certified by a physician, places the woman in danger of death) Acupuncture Bar...

  Financial highlights:


✅ All query results saved to 'rag_outputs/all_queries_results.json'


## 9. Summary and Usage Guide

### System Overview:
This notebook provides a complete RAG system for comparing insurance policies with two search options:

1. **Hybrid Search**: Combines semantic (embedding-based) and keyword (TF-IDF) search
2. **Hybrid Search + Reranking**: Adds intelligent reranking based on query coverage, financial relevance, and policy specificity

### Output Files:
- `rag_outputs/hybrid_search_results.txt`: Formatted for LLM prompting
- `rag_outputs/reranked_search_results.txt`: Formatted for LLM prompting with reranking scores
- `rag_outputs/*.json`: Structured data for programmatic access

### Next Steps for LLM Integration:
1. Use the formatted `.txt` files as context in your LLM prompts
2. Combine with policy analysis prompts like:
   - "Based on the search results, which policy offers better copay coverage?"
   - "Compare the deductibles and recommend the best policy for a family"
   - "Analyze the cost differences between these policies"

### Key Features:
- ✅ Policy-specific search and grouping
- ✅ Hybrid search combining semantic + keyword matching
- ✅ Advanced reranking with multiple relevance signals
- ✅ Financial data extraction for easy comparison
- ✅ LLM-ready formatted outputs
